# P05 — Estimación eficiente de representaciones de palabras en un espacio vectorial

## 1. Título y paper

**Paper:** *Efficient Estimation of Word Representations in Vector Space*  
**Autoría:** Tomas Mikolov, Kai Chen, Greg Corrado, Jeffrey Dean  
**Año y venue:** 2013 · arXiv:1301.3781 · ICLR 2013 (workshop)  
**Nivel:** L2 · **Motor:** `word2vec`  
**Ficha completa:** [`P05_word2vec`](../../papers/foundational/P05_word2vec/README.md)

**Hito:** El significado distribucional se vuelve barato: vectores densos entrenables sobre miles de millones de palabras.

- [arXiv:1301.3781](https://arxiv.org/abs/1301.3781)
- [arXiv:1310.4546 (muestreo negativo y frases)](https://arxiv.org/abs/1310.4546)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Representar palabras como identificadores dispersos (one-hot) impide medir similitud; los modelos neuronales de lenguaje previos eran demasiado costosos.
2. Ejecutar una implementación mínima de la propuesta: Dos arquitecturas log-lineales sin capa oculta —CBOW y skip-gram— que predicen contexto y producen vectores con estructura lineal.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- Bengio et al. (2003), modelo neuronal de lenguaje
- Harris (1954), hipótesis distribucional


## 4. Intuición

Dime con quién apareces y te diré qué significas. Si «rey» y «reina» aparecen rodeadas de las mismas palabras, sus vectores acabarán apuntando en direcciones parecidas — sin que nadie escriba jamás una definición.


## 5. Concepto mínimo

Skip-gram maximiza `log σ(v_c·u_o)` para pares (centro, contexto) reales y `log σ(−v_c·u_k)` para `k` pares negativos muestreados al azar.

El resultado es un espacio donde `coseno(a, b)` mide similitud distribucional y donde ciertas relaciones aparecen como desplazamientos aproximadamente constantes.


## 6. Código explicado

El motor entrena skip-gram con muestreo negativo sobre un corpus de 8 frases, en Python puro.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('word2vec', seed=7)['result']
print('vocabulario:', r['vocab_size'], '· dimensión:', r['dim'], '· pares:', r['training_pairs'])
show(r['neighbours'])

## 7. Predicción antes de ejecutar

1. ¿Quién estará más cerca de «rey»: «reina», «hombre» o «reino»?
2. ¿El resultado de `rey − hombre + mujer` será estable al cambiar la semilla?
3. Con solo 8 frases, ¿qué parte del resultado es señal y qué parte es ruido?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('word2vec', seed=semilla)['result']
    top = r['analogy_rey_menos_hombre_mas_mujer']
    print(f"semilla {semilla:>2} → 1º {top[0]['word']} ({top[0]['cos']}) · "
          f"2º {top[1]['word']} ({top[1]['cos']})")

## 9. Salida interpretable

«reina» sale primera en las tres semillas, con un coseno muy por encima del segundo lugar. Que el **primer** puesto sea estable y el **segundo** cambie es la lectura honesta: la señal fuerte se sostiene, la cola es ruido de un corpus diminuto.


## 10. Comentario pedagógico

La aritmética de analogías se popularizó como si el espacio codificara conceptos limpios. Trabajos posteriores mostraron que el resultado depende del protocolo (por ejemplo, de excluir del ranking las tres palabras de la consulta, como hace este código).


## 11. Error o anti-patrón deliberado

Anti-patrón: evaluar la analogía **sin excluir** las palabras de la consulta. El vecino más cercano acaba siendo la propia palabra de partida.


In [ ]:
print('Si no excluyes rey/hombre/mujer del ranking, «rey» suele ganar por su propio peso.')
print('El resultado parecería trivialmente correcto o trivialmente absurdo, según el caso.')
print('El protocolo de evaluación es parte del resultado, no un detalle de implementación.')

## 12. Corrección

El código del motor ya aplica la exclusión. Aquí se hace explícito el criterio:


In [ ]:
protocolo = {
    'consulta': 'rey - hombre + mujer',
    'excluidas_del_ranking': ['rey', 'hombre', 'mujer'],
    'metrica': 'coseno',
    'corpus': '8 frases (juguete)',
    'semillas_probadas': [1, 7, 42],
}
show(protocolo)

## 13. Desafío guiado

Comprueba si «calle» y «reino» quedan lejos entre sí: son las dos «familias» semánticas del corpus.


In [ ]:
r = run_paper_lab('word2vec', seed=7)['result']
show(r['neighbours']['calle'])

## 14. Desafío autónomo

Entrena embeddings sobre un corpus público en español de al menos 1 millón de palabras. Construye tu propio conjunto de 30 analogías y reporta accuracy top-1 y top-5, con la distribución de frecuencias de las palabras implicadas. Comenta el sesgo que encuentres.


## 15. Evidencia de aprendizaje

Guarda la tabla de vecinos, el resultado de la analogía en tres semillas y una frase sobre qué parte del resultado consideras evidencia y qué parte artefacto del corpus.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P05_word2vec/README.md) · evaluación formal: [`assessments/papers/P05_word2vec.md`](../../assessments/papers/P05_word2vec.md)


## 16. Cierre

Las palabras ya tienen geometría. Falta que un modelo produzca *secuencias* completas a partir de otras secuencias.


## 17. Conexión con el siguiente hito

- P06
- P11

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
